## Modelo Baseline
Será utilizado como modelo mais simples para efeito de comparação com os resultados do modelo Thompson Sampling. O modelo irá indicar a classe mais frequente para um dado cluster de tipo de clientes.

1. Carrega dataset, separa dados de treino e teste e mostra amostra dos dados

In [5]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# # ------------------------------------------------------------------
# Leitura dos dados processados
# # ------------------------------------------------------------------

input_path = Path("../data/processed/bank_marketing_processed.csv")
df = pd.read_csv(input_path)

print(f"Dataset carregado: {df.shape}")

# # ------------------------------------------------------------------
# Separação entre variáveis preditoras e alvo
# # ------------------------------------------------------------------

X = df.drop(columns=["y"])
y = df["y"]

# # ------------------------------------------------------------------
# Divisão treino e teste
# # ------------------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Treino: {X_train.shape}")
print(f"Teste : {X_test.shape}")

Dataset carregado: (41188, 25)
Treino: (32950, 24)
Teste : (8238, 24)


In [6]:
pd.set_option("display.max_columns", None) #visualizar todas as colunas
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,previous_contact,previous_success,age_group,financial_risk,engagement_score
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,senior,0,1
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,senior,0,1
2,37,services,married,high.school,no,yes,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,adult,1,1
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,adult,0,1
4,56,services,married,high.school,no,no,yes,telephone,may,mon,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0,senior,1,1


Treina e avalia o modelo 

In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ==========================================
# Escolha das variáveis que definem o perfil
# ==========================================

perfil = [
    "job",
    "education",
    "marital",
    "month",
    "previous_contact",
    "previous_success",
    "age_group",
    "financial_risk",
    "engagement_score",
]

# ==========================================
# Junta X e y de treino
# ==========================================

train = X_train.copy()
train["y"] = y_train.values

# Classe majoritária global
classe_global = train["y"].mode()[0]

# Classe mais frequente para cada perfil
baseline_por_perfil = (
    train
    .groupby(perfil)["y"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

# ==========================================
# Faz a previsão
# ==========================================

def prever(row):
    chave = tuple(row[col] for col in perfil)
    return baseline_por_perfil.get(chave, classe_global)

y_pred = X_test.apply(prever, axis=1)

# ==========================================
# Avaliação
# ==========================================

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_pred):.4f}")

Accuracy : 0.8795
Precision: 0.3502
Recall   : 0.0819
F1-score : 0.1328
ROC AUC  : 0.5313


| Métrica       |      Valor | Interpretação                                                                                                                                                                                                               |
| ------------- | ---------: | # ------------------------------------------------------------------# ------------------------------------------------------------------# --------------------------------------------------------------------------------------- |
| **Accuracy**  | **0,8795** | Alta, mas deve ser interpretada com cautela devido ao desbalanceamento da variável `y`. O baseline acerta aproximadamente **87,95%** das previsões, mas boa parte desse desempenho vem da predominância da classe negativa. |
| **Precision** | **0,3502** | Quando o baseline prevê **"Sim"**, ele acerta aproximadamente **35,02%** das vezes. Ou seja, cerca de 1 em cada 3 clientes classificados como conversão realmente converte.                                                 |
| **Recall**    | **0,0819** | Baixo. O baseline identifica apenas **8,19%** dos clientes que realmente convertem. Portanto, ele deixa de identificar aproximadamente **91,81%** das conversões.                                                           |
| **F1-score**  | **0,1328** | Baixo, refletindo o equilíbrio ruim entre Precision e Recall. O baseline apresenta dificuldade principalmente em recuperar os clientes da classe positiva.                                                                  |
| **ROC AUC**   | **0,5313** | Apenas ligeiramente acima de **0,5**, indicando baixa capacidade de discriminação entre clientes que convertem e não convertem. O baseline apresenta desempenho apenas marginalmente superior ao acaso.                     |


## Início dos testes de feature selection

In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ------------------------------------------------------------------
# Função para gerar as previsões do baseline
# ------------------------------------------------------------------

def avaliar_baseline(perfil):

    baseline_por_perfil = (
        train
        .groupby(perfil)["y"]
        .agg(lambda x: x.mode().iloc[0])
        .to_dict()
    )

    def prever(row):

        chave = tuple(
            row[col]
            for col in perfil
        )

        return baseline_por_perfil.get(
            chave,
            classe_global
        )

    y_pred = X_test.apply(
        prever,
        axis=1
    )

    return {
        "accuracy": accuracy_score(
            y_test,
            y_pred
        ),

        "precision": precision_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "recall": recall_score(
            y_test,
            y_pred,
            zero_division=0
        ),

        "f1": f1_score(
            y_test,
            y_pred,
            zero_division=0
        )
    }


# ------------------------------------------------------------------
# Performance utilizando todas as features
# ------------------------------------------------------------------

resultado_completo = avaliar_baseline(
    perfil
)

f1_completo = resultado_completo["f1"]


# ------------------------------------------------------------------
# Avaliar impacto de cada feature
# ------------------------------------------------------------------

resultados = []


for feature in perfil:

    perfil_reduzido = [
        col
        for col in perfil
        if col != feature
    ]

    resultado = avaliar_baseline(
        perfil_reduzido
    )

    queda_f1 = (
        f1_completo
        - resultado["f1"]
    )

    resultados.append({

        "feature": feature,

        "f1_completo": f1_completo,

        "f1_sem_feature": resultado["f1"],

        "queda_f1": queda_f1,

        "recall_sem_feature": resultado["recall"],

        "precision_sem_feature": resultado["precision"],

        "accuracy_sem_feature": resultado["accuracy"]
    })


# ------------------------------------------------------------------
# Tabela de importância
# ------------------------------------------------------------------

feature_importance = (
    pd.DataFrame(resultados)
    .sort_values(
        "queda_f1",
        ascending=False
    )
)

feature_importance["ganho_f1_ao_remover"] = (
    feature_importance["f1_sem_feature"]
    - feature_importance["f1_completo"]
)

print(
    feature_importance.to_string(
        index=False
    )
)

         feature  f1_completo  f1_sem_feature  queda_f1  recall_sem_feature  precision_sem_feature  accuracy_sem_feature  ganho_f1_ao_remover
previous_contact     0.132751        0.134380 -0.001629            0.082974               0.353211              0.879582             0.001629
previous_success     0.132751        0.138050 -0.005299            0.086207               0.346320              0.878733             0.005299
         marital     0.132751        0.141869 -0.009117            0.088362               0.359649              0.879582             0.009117
       age_group     0.132751        0.142606 -0.009855            0.087284               0.389423              0.881767             0.009855
             job     0.132751        0.152212 -0.019461            0.092672               0.425743              0.883710             0.019461
           month     0.132751        0.155203 -0.022452            0.094828               0.427184              0.883710             0.022452
      

A análise por ablação mostrou que a inclusão de todas as características não necessariamente melhora o baseline. As variáveis engagement_score, financial_risk e education apresentaram os maiores ganhos de F1 quando removidas, indicando que a granularidade excessiva dos perfis pode tornar a regra de classe majoritária menos robusta. Esse resultado não implica que essas variáveis sejam pouco relevantes para modelos mais sofisticados, como o Contextual Thompson Sampling, pois esses modelos conseguem aprender relações entre as características e a recompensa sem depender exclusivamente de agrupamentos exatos.

Retreinando modelo sem as features: education, financial_risk e engagement_score

In [10]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

# ==========================================
# Escolha das variáveis que definem o perfil
# ==========================================

perfil = [
    "job",
    "marital",
    "month",
    "previous_contact",
    "previous_success",
    "age_group",
]

# ==========================================
# Junta X e y de treino
# ==========================================

train = X_train.copy()
train["y"] = y_train.values

# Classe majoritária global
classe_global = train["y"].mode()[0]

# Classe mais frequente para cada perfil
baseline_por_perfil = (
    train
    .groupby(perfil)["y"]
    .agg(lambda x: x.mode().iloc[0])
    .to_dict()
)

# ==========================================
# Faz a previsão
# ==========================================

def prever(row):
    chave = tuple(row[col] for col in perfil)
    return baseline_por_perfil.get(chave, classe_global)

y_pred = X_test.apply(prever, axis=1)

# ==========================================
# Avaliação
# ==========================================

print(f"Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred):.4f}")
print(f"F1-score : {f1_score(y_test, y_pred):.4f}")
print(f"ROC AUC  : {roc_auc_score(y_test, y_pred):.4f}")

Accuracy : 0.8944
Precision: 0.6028
Recall   : 0.1832
F1-score : 0.2810
ROC AUC  : 0.5839


| Métrica       |      Valor | Interpretação                                                                                                                                                                                                                               |
| ------------- | ---------: | # ------------------------------------------------------------------# ------------------------------------------------------------------# ------------------------------------------------------------------------------------------------------- |
| **Accuracy**  | **0,8944** | Alta, mas deve ser interpretada com cautela devido ao desbalanceamento de `y`. O baseline acerta aproximadamente **89,44%** das previsões. Apesar disso, a Accuracy isoladamente não representa bem a capacidade de identificar conversões. |
| **Precision** | **0,6028** | Quando o baseline prevê **"Sim"**, ele acerta aproximadamente **60,28%** das vezes. Isso representa uma melhora significativa em relação à versão anterior, indicando maior precisão nas previsões positivas.                               |
| **Recall**    | **0,1832** | Ainda relativamente baixo, mas significativamente melhor. O baseline identifica **18,32%** dos clientes que realmente convertem, deixando de identificar aproximadamente **81,68%** das conversões.                                         |
| **F1-score**  | **0,2810** | Superior ao resultado anterior e indica um equilíbrio melhor entre Precision e Recall. Apesar da melhora, o baseline ainda apresenta limitações na identificação da classe positiva.                                                        |
| **ROC AUC**   | **0,5839** | Acima de **0,5**, indicando uma capacidade de discriminação superior ao acaso. Entretanto, o poder discriminativo ainda é moderado, deixando espaço para modelos mais sofisticados.                                                         |
